In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
import warnings
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn import set_config
set_config(display='diagram')

# Dataset: Bicicletas alugadas em Seoul 

Quantidade de bicicletas alugadas em Seoul, na Coreia do Sul, entre 2017 e 2018.

## Carregando o dataset e renomeando as colunas:

In [2]:
df = pd.read_excel("seoul_bike_data.xlsx", engine='openpyxl')

renome_colunas = {
    'DateTime': 'Data e hora',
    'Day': 'Dia',
    'Weekday': 'Dia da semana',
    'Hour': 'Hora',
    'Rented Bike Count': 'Bicicletas alugadas',
    'Temperature(°C)': 'Temperatura(°C)',
    'Humidity(%)': 'Humidade(%)',
    'Wind speed (m/s)': 'Velocidade do vento(m/s)',
    'Visibility (10m)': 'Visibilidade(10m)',
    'Dew point temperature(°C)': 'Ponto de orvalho(°C)',
    'Solar Radiation (MJ/m2)': 'Radiação solar(MJ/m2)',
    'Rainfall(mm)': 'Chuva(mm)',
    'Snowfall (cm)': 'Neve(cm)'
}

df = df.rename(columns=renome_colunas)
display(df)

,Data e hora,Dia,Dia da semana,Hora,Bicicletas alugadas,Temperatura(°C),Humidade(%),Velocidade do vento(m/s),Visibilidade(10m),Ponto de orvalho(°C),Radiação solar(MJ/m2),Chuva(mm),Neve(cm)
0,2017-01-12 00:00:00,12,5,0,254,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0
1,2017-01-12 01:00:00,12,5,1,204,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0
2,2017-01-12 02:00:00,12,5,2,173,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0
3,2017-01-12 03:00:00,12,5,3,107,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0
4,2017-01-12 04:00:00,12,5,4,78,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2018-12-31 19:00:00,31,2,19,163,0.0,31,2.2,2000,-15.1,0.0,0.0,0.0
8756,2018-12-31 20:00:00,31,2,20,161,-1.0,32,0.9,2000,-15.6,0.0,0.0,0.0
8757,2018-12-31 21:00:00,31,2,21,179,-1.6,35,1.0,2000,-15.1,0.0,0.0,0.0
8758,2018-12-31 22:00:00,31,2,22,155,-2.1,36,1.7,2000,-15.2,0.0,0.0,0.0


## Divisão do dataset em treino e teste:
Criei uma cópia do dataset anterior, depois dividi em treino e teste.

In [3]:
colunas = ['Dia da semana', 'Hora', 'Temperatura(°C)', 'Chuva(mm)', 'Neve(cm)'] 
X = df[colunas]
y = df['Bicicletas alugadas'] 

df = df[df['Bicicletas alugadas'] <= 1500]

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Tamanho do conjunto de treino: {len(X_treino)} ({len(X_treino)/len(df)*100:.1f}%)")
print(f"Tamanho do conjunto de teste: {len(X_teste)} ({len(X_teste)/len(df)*100:.1f}%)")

Tamanho do conjunto de treino: 6570 (86.1%)
Tamanho do conjunto de teste: 2190 (28.7%)


### Preparação dos dados (limpeza, transformação):

In [4]:
df_selecao = df.iloc[:, [1, 2, 3, 4, 5, 7, 11, 12]] 
display(df_selecao)

,Dia,Dia da semana,Hora,Bicicletas alugadas,Temperatura(°C),Velocidade do vento(m/s),Chuva(mm),Neve(cm)
0,12,5,0,254,-5.2,2.2,0.0,0.0
1,12,5,1,204,-5.5,0.8,0.0,0.0
2,12,5,2,173,-6.0,1.0,0.0,0.0
3,12,5,3,107,-6.2,0.9,0.0,0.0
4,12,5,4,78,-6.0,2.3,0.0,0.0
...,...,...,...,...,...,...,...,...
8755,31,2,19,163,0.0,2.2,0.0,0.0
8756,31,2,20,161,-1.0,0.9,0.0,0.0
8757,31,2,21,179,-1.6,1.0,0.0,0.0
8758,31,2,22,155,-2.1,1.7,0.0,0.0


### Treinamento do modelo com XGBRegressor:

In [5]:
modelo = XGBRegressor(n_estimators=200, learning_rate=0.1)
modelo.fit(X_treino, y_treino)

y_previsao = modelo.predict(X_teste)

print("Colunas do modelo de previsão:", modelo.feature_names_in_)

Colunas do modelo de previsão: ['Dia da semana' 'Hora' 'Temperatura(°C)' 'Chuva(mm)' 'Neve(cm)']


## Previsões:
### 1 - Prever a quantidade de bicicletas alugadas para um domingo às 13h, com 25 ºC e sem chuva.

In [6]:
novos_dados = pd.DataFrame({         
    'Dia da semana': [1],       
    'Hora': [13], 
    'Temperatura(°C)': [25],       
    'Chuva(mm)': [0],           
    'Neve(cm)': [0],            
})

previsao = modelo.predict(novos_dados)

print(f"Previsão de bicicletas alugadas: {int(previsao[0])} unidades")

Previsão de bicicletas alugadas: 1173 unidades


### 2 - Prever a quantidade de bicicletas alugadas para um domingo às 13h, com 25 ºC e com chuva.

In [7]:
novos_dados = pd.DataFrame({         
    'Dia da semana': [1],       
    'Hora': [13], 
    'Temperatura(°C)': [25],       
    'Chuva(mm)': [1],           
    'Neve(cm)': [0],            
})

previsao = modelo.predict(novos_dados)

print(f"Previsão de bicicletas alugadas: {int(previsao[0])} unidades")

Previsão de bicicletas alugadas: 556 unidades


### 3 - Prever a quantidade de bicicletas alugadas para uma quinta às 7h, com -4 ºC e com neve.

In [8]:
novos_dados = pd.DataFrame({         
    'Dia da semana': [5],       
    'Hora': [7], 
    'Temperatura(°C)': [-4],       
    'Chuva(mm)': [0],           
    'Neve(cm)': [1],            
})

previsao = modelo.predict(novos_dados)

print(f"Previsão de bicicletas alugadas: {int(previsao[0])} unidades")

Previsão de bicicletas alugadas: 246 unidades


## Carregando o dataset novamente:

E dividindo os dados em treino e teste novamente.

In [9]:
df_copia = df.copy()
colunas = ['Dia da semana', 'Hora', 'Temperatura(°C)', 'Chuva(mm)', 'Neve(cm)'] 
X = df_copia[colunas]
y = df_copia['Bicicletas alugadas'] 

df_copia = df_copia[df_copia['Bicicletas alugadas'] <= 1500]

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Tamanho do conjunto de treino: {len(X_treino)} ({len(X_treino)/len(df)*100:.1f}%)")
print(f"Tamanho do conjunto de teste: {len(X_teste)} ({len(X_teste)/len(df)*100:.1f}%)")

Tamanho do conjunto de treino: 5721 (75.0%)
Tamanho do conjunto de teste: 1907 (25.0%)


## Organizando os dados para criação do Pipeline:

Dividindo os tipos de colunas em numéricas e categóricas.

In [10]:
colunas_numericas = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
colunas_categoricas = X.select_dtypes(include=['object', 'category']).columns.tolist()
 
transformador_numerico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler())  
])

transformador_categorico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  
    ('onehot', OneHotEncoder(handle_unknown='ignore'))  
])

preprocessador = ColumnTransformer(
    transformers=[
        ('num', transformador_numerico, colunas_numericas),
        ('cat', transformador_categorico, colunas_categoricas)
    ])

print("Organização dos dados concluída!")

Organização dos dados concluída!


## Criando o Pipeline:

In [11]:
pipeline = Pipeline([('preprocessor', preprocessador),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))]).fit(X_treino, y_treino)

y_pred = pipeline.predict(X_teste)
display(f'Resultados de y_pred: {y_pred}')

display("Passos do pipeline:")
display(pipeline)


'Resultados de y_pred: [309.59       169.91952381  35.50333333 ... 283.24       925.08428571\n 495.835     ]'

'Passos do pipeline:'

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Dia da semana', 'Hora',
                                                   'Temperatura(°C)',
                                                   'Chuva(mm)', 'Neve(cm)']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  [])])),
                ('regressor', RandomForestRegressor(random_state=42))])

## Previsões:

### 1 - Prever a quantidade de bicicletas alugadas para um domingo às 13h, com 25 ºC e sem chuva.

In [12]:
novos_dados = pd.DataFrame({         
    'Dia da semana': [1],       
    'Hora': [13], 
    'Temperatura(°C)': [25],       
    'Chuva(mm)': [0],           
    'Neve(cm)': [0],            
})

previsao = pipeline.predict(novos_dados)

print(f"Previsão de bicicletas alugadas: {int(previsao[0])} unidades")

Previsão de bicicletas alugadas: 1212 unidades


### 2 - Prever a quantidade de bicicletas alugadas para um domingo às 13h, com 25 ºC e com chuva.

In [13]:
novos_dados = pd.DataFrame({         
    'Dia da semana': [1],       
    'Hora': [13], 
    'Temperatura(°C)': [25],       
    'Chuva(mm)': [1],           
    'Neve(cm)': [0],            
})

previsao = pipeline.predict(novos_dados)

print(f"Previsão de bicicletas alugadas: {int(previsao[0])} unidades")

Previsão de bicicletas alugadas: 173 unidades


### 3 - Prever a quantidade de bicicletas alugadas para uma quinta às 7h, com -4 ºC e com neve.

In [14]:
novos_dados = pd.DataFrame({         
    'Dia da semana': [5],       
    'Hora': [7], 
    'Temperatura(°C)': [-4],       
    'Chuva(mm)': [0],           
    'Neve(cm)': [1],            
})

previsao = pipeline.predict(novos_dados)

print(f"Previsão de bicicletas alugadas: {int(previsao[0])} unidades")

Previsão de bicicletas alugadas: 254 unidades


## Calculando as métricas:

In [15]:
mae = mean_absolute_error(y_teste, y_pred)
r2 = r2_score(y_teste, y_pred)

print(f"\nMétricas de avaliação:")
print("MAE:", mae)
print("R²:", r2)




Métricas de avaliação:
MAE: 171.38773290077341
R²: 0.5746939691237212


## Conclusões Finais:

**MAE = 171.38:**<br>
Quanto mais próximo de 0, melhor.<br>
Significa que o modelo erra, em média, por 171 unidades.<br> 
Isso indica que o modelo erra muito.

**R² = 0.574:**<br>
Quanto mais próximo de 1, melhor.<br>
Significa que 57.4% da variabilidade dos dados é explicada pelo modelo.<br>
Isso indica que o modelo é razoável, mas pode ser melhorado.<br>

**O desempenho do modelo é razoável, mas pode ser melhorado significativamente.**<br>

**Possíveis melhorias:**<br>
Experimentar outros modelos.<br>
Tratamento de outliers.<br>
Usar técnicas de hiperparâmetros.